# Notebook 03 — NLP Urgency & Risk Scoring Engine
### Phishing Email Forensic Analyzer & Link Defanger (Part 3 of 4)

This notebook:
1. Mounts Drive and adds `modules/` to `sys.path`.
2. Writes **`nlp_analyzer.py`** — keyword/heuristic detection of
   urgency, psychological manipulation, credential-harvesting, and
   threat language.
3. Writes **`risk_scorer.py`** — combines `link_defanger` +
   `nlp_analyzer` output into a single weighted **0–100 risk score**.
4. Runs an end-to-end scoring test on mock phishing text.

> **Cybersecurity context — social engineering signals:** Phishing
> succeeds by manufacturing urgency and fear ("your account will be
> **suspended**", "**verify within 24 hours**") to short-circuit a
> victim's normal skepticism, or by promising reward ("**you've been
> selected**") to lower their guard. These patterns are well-documented
> in security-awareness training and are exactly what a human analyst
> scans for first — we're encoding that same triage instinct into
> keyword categories.
>
> **Cybersecurity context — weighted scoring:** No single signal alone
> proves an email is malicious (urgent language also appears in
> legitimate shipping/billing notices). A weighted composite score,
> with domain-mismatch and credential-harvesting weighted heaviest,
> better reflects real analyst judgment than any single yes/no rule.

## 1. Mount Drive & Configure Import Path

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/Phishing_Analyzer_Project"
MODULES_DIR = os.path.join(PROJECT_ROOT, "modules")
os.makedirs(MODULES_DIR, exist_ok=True)
sys.path.append(MODULES_DIR)
print("Modules dir on sys.path:", MODULES_DIR in sys.path)

Mounted at /content/drive
Modules dir on sys.path: True


## 2. Write `nlp_analyzer.py` to Drive

In [2]:
%%writefile /content/drive/MyDrive/Phishing_Analyzer_Project/modules/nlp_analyzer.py
"""
nlp_analyzer.py
----------------
  1. Urgency               - manufactured time pressure
  2. Credential harvesting - explicit requests to (re)enter secrets
  3. Psychological          - generic/impersonal framing, rewards, "click here"
     manipulation
  4. Threats                - fear of legal/financial/account consequences

Keyword lists are intentionally simple and auditable -- an analyst can
read this file top to bottom and understand exactly why an email was
flagged, which matters for a forensic tool that will influence
real decisions.
"""

URGENCY_KEYWORDS = [
    "urgent", "immediately", "act now", "action required",
    "within 24 hours", "expires today", "final notice",
    "account suspended", "account will be closed", "limited time",
    "respond immediately",
]

CREDENTIAL_HARVEST_KEYWORDS = [
    "verify your account", "confirm your password", "click here to verify",
    "update your payment information", "login to confirm",
    "reset your password", "verify your identity", "enter your credentials",
    "confirm your identity",
]

MANIPULATION_KEYWORDS = [
    "dear customer", "dear valued customer", "winner", "congratulations",
    "you have been selected", "claim your prize", "click here",
    "download the attachment", "wire transfer", "gift card",
]

THREAT_KEYWORDS = [
    "legal action", "account will be terminated",
    "unauthorized access detected", "suspicious activity",
    "your account has been locked", "failure to comply",
]

CATEGORIES = {
    "urgency": URGENCY_KEYWORDS,
    "credential_harvesting": CREDENTIAL_HARVEST_KEYWORDS,
    "psychological_manipulation": MANIPULATION_KEYWORDS,
    "threats": THREAT_KEYWORDS,
}


def analyze_text(text):
    """
    Scan `text` (typically subject + body) for phishing-associated
    language across the four categories above.

    Returns:
        {
          "matches": {category: [matched keywords...], ...},
          "total_hits": int,
          "matched_phrases": [all matched keywords, flat list],
        }
    """
    if not text:
        return {"matches": {c: [] for c in CATEGORIES}, "total_hits": 0, "matched_phrases": []}

    lowered = text.lower()

    matches = {}
    matched_phrases = []
    for category, keywords in CATEGORIES.items():
        hits = [kw for kw in keywords if kw in lowered]
        matches[category] = hits
        matched_phrases.extend(hits)

    return {
        "matches": matches,
        "total_hits": len(matched_phrases),
        "matched_phrases": matched_phrases,
    }

Overwriting /content/drive/MyDrive/Phishing_Analyzer_Project/modules/nlp_analyzer.py


## 3. Write `risk_scorer.py` to Drive

In [3]:
%%writefile /content/drive/MyDrive/Phishing_Analyzer_Project/modules/risk_scorer.py
"""
risk_scorer.py
---------------
Cybersecurity note: combines link-domain signals (link_defanger) and
language signals (nlp_analyzer) into one weighted 0-100 score. Weight
rationale, in order of real-world reliability as a phishing indicator:

  Domain mismatch          up to 45 pts  (near-certain phishing signal)
  Credential harvesting    up to 25 pts  (legit orgs rarely ask via email)
  Urgency language         up to 15 pts  (common in phishing AND some legit mail)
  Threatening language     up to 10 pts  (supporting signal)
  Generic manipulation      up to 5 pts  (weak/supporting signal)
                           ------------
                           100 pts max

Bands:
  0-34   -> SAFE
  35-69  -> SUSPICIOUS
  70-100 -> DANGEROUS
"""


def compute_risk_score(nlp_result, link_results):
    """
    Parameters
    ----------
    nlp_result : dict returned by nlp_analyzer.analyze_text()
    link_results : list[dict] returned by link_defanger.analyze_links()

    Returns
    -------
    dict: {"score": int, "category": str, "reasons": list[str]}
    """
    score = 0
    reasons = []

    link_results = link_results or []
    nlp_result = nlp_result or {}
    matches = nlp_result.get("matches", {})

    # --- Link-based scoring (max 45 pts) ---
    mismatched = [l for l in link_results if l.get("domain_mismatch")]
    if mismatched:
        mismatch_points = min(45, 15 * len(mismatched))
        score += mismatch_points
        reasons.append(
            f"{len(mismatched)} link(s) point to a domain different from the sender's claimed domain."
        )

    # --- NLP-based scoring (max 55 pts total across categories) ---
    credential_hits = matches.get("credential_harvesting", [])
    if credential_hits:
        pts = min(25, 10 * len(credential_hits))
        score += pts
        reasons.append(f"Credential-harvesting language detected: {', '.join(credential_hits)}")

    urgency_hits = matches.get("urgency", [])
    if urgency_hits:
        pts = min(15, 5 * len(urgency_hits))
        score += pts
        reasons.append(f"Urgency language detected: {', '.join(urgency_hits)}")

    threat_hits = matches.get("threats", [])
    if threat_hits:
        pts = min(10, 5 * len(threat_hits))
        score += pts
        reasons.append(f"Threatening language detected: {', '.join(threat_hits)}")

    manipulation_hits = matches.get("psychological_manipulation", [])
    if manipulation_hits:
        pts = min(5, 2 * len(manipulation_hits))
        score += pts
        reasons.append(f"Generic manipulation phrasing detected: {', '.join(manipulation_hits)}")

    score = max(0, min(100, score))

    if score >= 70:
        category = "DANGEROUS"
    elif score >= 35:
        category = "SUSPICIOUS"
    else:
        category = "SAFE"

    if not reasons:
        reasons.append("No significant phishing indicators detected.")

    return {"score": score, "category": category, "reasons": reasons}

Overwriting /content/drive/MyDrive/Phishing_Analyzer_Project/modules/risk_scorer.py


## 4. Self-Test — End-to-End Scoring with Mock Inputs

In [4]:
!pip install tldextract
import nlp_analyzer, link_defanger, risk_scorer
import importlib
importlib.reload(nlp_analyzer)
importlib.reload(link_defanger)
importlib.reload(risk_scorer)

mock_subject = "Urgent: Your account has been suspended"
mock_body = '''
Dear Customer,

Your account has been suspended due to suspicious activity.
You must verify your account and confirm your password within 24 hours
or legal action may be taken.

Click here to verify: http://paypa1-verify.com/login
'''
mock_sender = "service@paypal.com"
full_text = mock_subject + "\n" + mock_body

nlp_result = nlp_analyzer.analyze_text(full_text)
link_results = link_defanger.analyze_links(mock_body, mock_sender)
risk = risk_scorer.compute_risk_score(nlp_result, link_results)

print("NLP matches:", nlp_result["matches"])
print("\nLink analysis:")
for l in link_results:
    print(" ", l)

print(f"\n=== RISK SCORE: {risk['score']}/100  ({risk['category']}) ===")
for r in risk["reasons"]:
    print(" -", r)

print("\n--- Sanity check: a benign email should score low ---")
benign_text = "Hi team, please find attached the quarterly report. Thanks!"
benign_nlp = nlp_analyzer.analyze_text(benign_text)
benign_links = link_defanger.analyze_links(benign_text, "colleague@company.com")
benign_risk = risk_scorer.compute_risk_score(benign_nlp, benign_links)
print(f"Benign email risk: {benign_risk['score']}/100 ({benign_risk['category']})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 4.5 MB/s eta 0:00:00
NLP matches: {'urgency': ['urgent', 'within 24 hours'], 'credential_harvesting': ['verify your account', 'confirm your password', 'click here to verify'], 'psychological_manipulation': ['dear customer', 'click here'], 'threats': ['legal action', 'suspicious activity']}

Link analysis:
  {'original_url': 'http://paypa1-verify.com/login', 'defanged_url': 'hxxp://paypa1-verify[.]com/login', 'link_domain': 'paypa1-verify.com', 'sender_domain': 'paypal.com', 'domain_mismatch': True}

=== RISK SCORE: 64/100  (SUSPICIOUS) ===
 - 1 link(s) point to a domain different from the sender's claimed domain.
 - Credential-harvesting language detected: verify your account, confirm your password, click here to verify
 - Urgency language detected: urgent, within 24 hours
 - Threatening language detected: legal action, suspicious activity
 - Generic manipulation phrasing detected: dear customer, click here

--- Sanity check: a